# Few-Shot Classification — UVA Kimi K2.5

Companion to `zeroshot_kimik2_uva.ipynb`. Same UVA Kimi K2.5 endpoint and prompt base, but now few-shot:

- **Rows 21–40** of `labeled_dataset_new.xlsx` are used as the labeled few-shot examples shown to the model (using their existing `Label` / `label_reason`).
- **Rows 1–20** are the ones actually classified, so you can directly compare this run's `predicted_label` against the zero-shot run's `predicted_label` on the *same* 20 posts.

Same API key setup as before — `.env` file (or shell env var) with `UVARC_GenAI_API=sk-...` next to this notebook.

In [ ]:
!pip3 install httpx pandas openpyxl --break-system-packages

In [ ]:
"""
Few-Shot Kimi K2.5 classification (UVA RC OpenWebUI).
Examples: rows 21-40 of labeled_dataset_new.xlsx
Test set: rows 1-20 of labeled_dataset_new.xlsx
"""

import os
import re
import sys
import json
import time
import httpx
import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────
INPUT_PATH   = "labeled_dataset_new.xlsx"   # edit path if it's elsewhere
INPUT_SHEET  = 0                             # first sheet; change if needed

TEST_ROWS_START,    TEST_ROWS_END    = 0, 20    # rows 1-20 (0-indexed slice)
FEWSHOT_ROWS_START,  FEWSHOT_ROWS_END = 20, 40   # rows 21-40 (0-indexed slice)

OUTPUT_PATH  = "labeled_dataset_new_kimik2_fewshot_first20.csv"

TEXT_COL  = "text"
LABEL_COL = "Label"
ID_COL    = "id"

UVARC_BASE_URL = "https://open-webui.rc.virginia.edu/api"
UVARC_CHAT_ENDPOINT = f"{UVARC_BASE_URL}/chat/completions"
MODEL = "Kimi K2.5"

TEMPERATURE = 0.0
MAX_TOKENS  = 2000        # reasoning model burns tokens on chain-of-thought before output
REQUEST_DELAY = 0.5
MAX_RETRIES = 5
DEBUG_FIRST_N = 2         # print raw response for the first N calls so you can sanity-check parsing


# ── .env loader (same pattern as before) ─────────────────────────────────────
def _load_env_file():
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()

    for candidate in [os.path.join(script_dir, ".env"), os.path.join(os.getcwd(), ".env")]:
        if os.path.exists(candidate):
            print(f"  Loading .env from: {candidate}")
            with open(candidate) as f:
                for line in f:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    line = re.sub(r"^export\s+", "", line)
                    if "=" in line:
                        k, _, v = line.partition("=")
                        k = k.strip()
                        v = v.strip().strip('"').strip("'")
                        if k and k not in os.environ:
                            os.environ[k] = v
            return candidate
    return None


_found = _load_env_file()
if not _found:
    print("  (No .env file found — falling back to shell environment)")

UVARC_API_KEY = os.environ.get("UVARC_GenAI_API")
if not UVARC_API_KEY:
    sys.exit(
        "\nERROR: UVARC_GenAI_API not found.\n"
        "  Create a .env file next to this notebook containing:\n"
        "    UVARC_GenAI_API=sk-your-key-here\n"
    )
print("API key loaded (starts with):", UVARC_API_KEY[:8] + "...")

In [ ]:
# ── Prompt templates (zero-shot header + few-shot wrapper, same as original) ─
BASE_HEADER = """You are an expert in labeling burnout-related Reddit posts from cybersecurity professionals.

Classify the post into ONE of the following categories:

0 (out-of-scope): Not about work stress/burnout at all — pure technical questions, news, product discussions, memes, or other unrelated content.

1-9 (in-scope): The post relates to work-related burnout or stress in one of these specific ways:
  1 = Work-related burnout / chronic stress / workload / work pressure (job demands, long hours, on-call, exhaustion from work, etc.)
  2 = Toughest situation — could be work related; a hint of possible stress, work-related frustration, mental exhaustion, or depression
  3 = Imposter syndrome in work life; not knowing something work-related; feeling like an outsider/pariah
  4 = PTSD, mental health disorder/illness, bipolar disorder, ADHD, schizophrenia
  5 = De-stress / stress relief / trying not to become overwhelmed
  6 = Work-life balance, work-related health, workload balance
  7 = Job search / interview / hard time getting a job / job change / job confusion / job tasks feel slow or not getting it / not feeling good enough about job / job market burnout / trust issues at work-ethical stress / toxic environment
  8 = Anxiety
  9 = Staying motivated, learning new things

IMPORTANT — final label rule: Categories 1 through 9 are ALL in-scope. Regardless of which specific subcategory (1-9) applies, the final "label" field must be 1. Only use "label": 0 for posts that are genuinely out-of-scope (category 0). The "subcategory" field is where you record which specific number (0-9) applies — this is for reasoning/audit purposes and is separate from "label".

Emphasis & Caution: Hypothetical or purely future/imaginary burnout scenarios where the poster is not describing their own past or present experience should lean toward out-of-scope (0) unless another in-scope category clearly applies (e.g. job search anxiety about a future interview is still in-scope under category 7/8).

Respond ONLY in this exact JSON format (no other text, no markdown, no code fences):
{{
  "subcategory": 0-9,
  "label": 0 or 1,
  "label_reason": "1-2 sentences citing the specific language in the post that supports this category"
}}"""


def build_few_shot_prompt(text, fewshot_examples_text):
    return f"""{BASE_HEADER}

Here are some labeled examples:

{fewshot_examples_text}

Now classify the following post:
\"\"\"
{text}
\"\"\"

JSON:"""


def create_fewshot_examples_text(fewshot_df):
    """
    Uses the existing label_reason text from labeled_dataset_new.xlsx as the
    reasoning field in each example. If a subcategory number appears in
    parentheses in label_reason (e.g. "Job search (7): ..."), it's extracted;
    otherwise subcategory falls back to the collapsed label (0 or 1).
    """
    examples = []
    for _, row in fewshot_df.iterrows():
        label = 1 if row[LABEL_COL] == 1 else 0
        reason = str(row.get("label_reason", "")).strip()

        subcat_match = re.search(r"\((\d)(?:/\d)?\)", reason)
        subcategory = int(subcat_match.group(1)) if subcat_match else label

        reason_escaped = reason.replace("\\", "\\\\").replace('"', '\\"')

        example_json = (
            f'{{"subcategory": {subcategory}, "label": {label}, '
            f'"label_reason": "{reason_escaped}"}}'
        )
        examples.append(
            f"Post:\n\"\"\"\n{row[TEXT_COL]}\n\"\"\"\nJSON:\n{example_json}"
        )
    return "\n\n".join(examples)

In [ ]:
# ── UVA Kimi K2.5 call (raw httpx, defensive parsing — identical to zero-shot notebook) ─
_call_counter = {"n": 0}


def _extract_text_from_response(raw_text):
    raw_text = raw_text.strip()

    # 1) Plain JSON body, OpenAI-style: {"choices":[{"message":{"content":...}}]}
    try:
        data = json.loads(raw_text)
        choice = data["choices"][0]
        if "message" in choice and "content" in choice["message"]:
            return choice["message"]["content"]
        if "text" in choice:
            return choice["text"]
    except Exception:
        pass

    # 2) SSE stream: lines like "data: {...}\n\n", ending in "data: [DONE]"
    if "data:" in raw_text:
        pieces = []
        for line in raw_text.splitlines():
            line = line.strip()
            if not line.startswith("data:"):
                continue
            payload = line[len("data:"):].strip()
            if payload == "[DONE]" or not payload:
                continue
            try:
                chunk = json.loads(payload)
                delta = chunk["choices"][0].get("delta", {})
                content = delta.get("content")
                if content:
                    pieces.append(content)
            except Exception:
                continue
        if pieces:
            return "".join(pieces)

    return None


def call_kimi(prompt, max_retries=MAX_RETRIES):
    headers = {
        "Authorization": f"Bearer {UVARC_API_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stream": False,
    }

    for attempt in range(max_retries):
        try:
            with httpx.Client(timeout=120.0) as client:
                resp = client.post(UVARC_CHAT_ENDPOINT, headers=headers, json=body)

            _call_counter["n"] += 1
            if _call_counter["n"] <= DEBUG_FIRST_N:
                print(f"\n--- RAW RESPONSE (call {_call_counter['n']}, status {resp.status_code}) ---")
                print(resp.text[:2000])
                print("--- END RAW RESPONSE ---\n")

            if resp.status_code == 401:
                sys.exit("ERROR: 401 Unauthorized — check that UVARC_GenAI_API is a valid, non-expired key.")

            resp.raise_for_status()
            text = _extract_text_from_response(resp.text)
            if text is not None:
                return text.strip()

            print("  Could not parse response into text — see raw output above. Retrying...")
            wait = 10
        except httpx.HTTPStatusError as e:
            err = str(e).lower()
            wait = 10 * (2 ** attempt) if ("rate" in err or "limit" in err or "503" in err) else 10
            print(f"  HTTP error: {e} — waiting {wait}s (attempt {attempt+1}/{max_retries})")
        except Exception as e:
            wait = 10
            print(f"  Error: {e} — waiting {wait}s (attempt {attempt+1}/{max_retries})")

        time.sleep(wait)

    print("  Max retries exceeded — defaulting to empty response")
    return ""


def parse_output(output):
    cleaned = re.sub(r"```(?:json)?", "", output).strip()
    try:
        match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if match:
            data = json.loads(match.group())
            label = int(data.get("label", 0))
            label = 1 if label != 0 else 0
            subcategory = data.get("subcategory", label)
            reason = str(data.get("label_reason", "")).strip()
            return label, subcategory, reason
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    fallback_match = re.search(r"[01]", cleaned)
    label = int(fallback_match.group()) if fallback_match else 0
    return label, label, "PARSE_FAILURE — raw output could not be parsed as JSON"

In [ ]:
# ── Load data, build few-shot examples from rows 21-40, classify rows 1-20 ──
df = pd.read_excel(INPUT_PATH, sheet_name=INPUT_SHEET)
print(f"Loaded {len(df)} rows total.")

test_df    = df.iloc[TEST_ROWS_START:TEST_ROWS_END].copy()
fewshot_df = df.iloc[FEWSHOT_ROWS_START:FEWSHOT_ROWS_END].copy()

print(f"Test set (classified this run): rows {TEST_ROWS_START+1}-{TEST_ROWS_END} → {len(test_df)} posts")
print(f"Few-shot examples (shown to model): rows {FEWSHOT_ROWS_START+1}-{FEWSHOT_ROWS_END} → {len(fewshot_df)} posts")

fewshot_text = create_fewshot_examples_text(fewshot_df)

results = []
for i, row in test_df.iterrows():
    text = str(row[TEXT_COL])
    prompt = build_few_shot_prompt(text, fewshot_text)

    raw_output = call_kimi(prompt)
    predicted_label, predicted_subcategory, label_reason = parse_output(raw_output)

    true_label = row.get(LABEL_COL, None)
    results.append({
        ID_COL: row.get(ID_COL, i),
        TEXT_COL: text,
        "true_label": true_label,
        "predicted_label": predicted_label,
        "predicted_subcategory": predicted_subcategory,
        "label_reason": label_reason,
        "match": (predicted_label == true_label) if pd.notna(true_label) else None,
        "raw_output": raw_output,
    })

    print(f"[{len(results)}/{len(test_df)}] true={true_label} pred={predicted_label} (subcat {predicted_subcategory})")
    time.sleep(REQUEST_DELAY)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_PATH, index=False)

n_matches = results_df["match"].sum()
n_scored = results_df["match"].notna().sum()
print(f"\nSaved: {OUTPUT_PATH}")
print(f"Agreement with human Label on this batch: {n_matches}/{n_scored}")